# Distance Metrics for Ranking Lists

In [1]:
import numpy as np

## Standard Distance

In [2]:
def kendall_tau(a: np.array, b: np.array) -> int:
    dist = 0
    for i in range(a.shape[0]):
        for j in range(i + 1, a.shape[0]):
            dist += (a[i] > a[j]) != (b[i] > b[j])
    return dist

In [3]:
arr_a = np.array([1, 2, 3, 4, 5])
arr_b = np.array([2, 1, 4, 5, 3])

kendall_tau(arr_a, arr_b)

np.int64(3)

In [4]:
arr_a = np.array([1, 2, 3, 4, 5])
arr_b = np.array([2, 5, 4, 1, 3])

kendall_tau(arr_a, arr_b)

np.int64(6)

In [5]:
arr_a = np.array([1, 2, 3, 4, 5])
arr_b = np.array([5, 1, 4, 2, 3])

kendall_tau(arr_a, arr_b)

np.int64(6)

## Weighted Distance

In [6]:
def kendall_tau_weighted(a: np.array, b: np.array, w: np.array) -> int:
    dist = 0
    for i in range(a.shape[0]):
        for j in range(i + 1, a.shape[0]):
            dist += w[i] * w[j] * ((a[i] > a[j]) != (b[i] > b[j]))
    return dist

In [7]:
arr_a = np.array([1, 2, 3, 4, 5])
arr_b = np.array([2, 1, 4, 5, 3])
weights = np.array([1.0, 0.8, 0.6, 0.4, 0.2])

kendall_tau_weighted(arr_a, arr_b, weights)

np.float64(1.0)

In [8]:
arr_a = np.array([1, 2, 3, 4, 5])
arr_b = np.array([2, 5, 4, 1, 3])
weights = np.array([1.0, 0.8, 0.6, 0.4, 0.2])

kendall_tau_weighted(arr_a, arr_b, weights)

np.float64(1.7200000000000002)

In [9]:
arr_a = np.array([1, 2, 3, 4, 5])
arr_b = np.array([5, 1, 4, 2, 3])
weights = np.array([1.0, 0.8, 0.6, 0.4, 0.2])

kendall_tau_weighted(arr_a, arr_b, weights)

np.float64(2.36)

## Top-Item Recovery

In [10]:
def kendall_tau_top(a: np.array, b: np.array) -> int:
    dist = 0
    b_dict = {b[i]: i for i in range(b.shape[0])}

    for j in range(1, a.shape[0]):
        dist += (0 > j) != b_dict[a[0]] > b_dict[a[j]]
    
    return dist

In [11]:
arr_a = np.array([1, 2, 3, 4, 5])
arr_b = np.array([2, 1, 4, 5, 3])

kendall_tau_top(arr_a, arr_b)

1

In [12]:
arr_a = np.array([1, 2, 3, 4, 5])
arr_b = np.array([2, 5, 4, 1, 3])

kendall_tau_top(arr_a, arr_b)

3

In [13]:
arr_a = np.array([1, 2, 3, 4, 5])
arr_b = np.array([5, 1, 4, 2, 3])

kendall_tau_top(arr_a, arr_b)

1

## Top-K Kendall Distance

In [14]:
import math

def top_k_kendall_tau(a: np.array, b: np.array, k: int, p: float) -> int:
    dist = 0
    p_dist = 0
    a_dict = {a[i]: i for i in range(k)}
    b_dict = {b[i]: i for i in range(k)}
    remaining_b = set(b_dict.keys()) - set(a_dict.keys())
    #print(remaining_b)
    for i in range(k):
        for j in range(i + 1, k):
            if a[i] not in b_dict and a[j] not in b_dict:
                p_dist += p
            else:
                if a[i] not in b_dict:
                    b_later = True
                elif a[j] not in b_dict:
                    b_later = False
                else:
                    b_later = b_dict[a[i]] > b_dict[a[j]]
                a_later = (i > j)
                #print(i, j, a_later, b_later)
                dist += a_later != b_later
        for item in remaining_b:
            if a[i] not in b_dict:
                b_later = True
            else:
                b_later = b_dict[a[i]] > b_dict[item]
            a_later = False
            #print(i, b_dict[item], a_later, b_later)
            dist += a_later != b_later
    
    p_dist += math.comb(len(remaining_b), 2) * p
    return dist + p_dist

In [15]:
arr_a = np.array([1, 2, 3, 4, 5])
arr_b = np.array([2, 1, 4, 5, 3])
k = 3
p = 0.5123

top_k_kendall_tau(arr_a, arr_b, k, p)

2.0

In [16]:
arr_a = np.array([1, 2, 3, 4, 5])
arr_b = np.array([2, 5, 4, 1, 3])
k = 3
p = 0.5123

top_k_kendall_tau(arr_a, arr_b, k, p)

6.0245999999999995

In [17]:
arr_a = np.array([1, 2, 3, 4, 5])
arr_b = np.array([5, 1, 4, 2, 3])
k = 3
p = 0.5123

top_k_kendall_tau(arr_a, arr_b, k, p)

6.0245999999999995

In [18]:
arr_a = np.array([1, 2, 3])
arr_b = np.array([3, 2, 1])
k = 2
p = 0.5123

top_k_kendall_tau(arr_a, arr_b, k, p)

3.0

In [19]:
arr_a = np.array([1, 2, 3])
arr_b = np.array([2, 1, 3])
k = 2
p = 0.5123

top_k_kendall_tau(arr_a, arr_b, k, p)

1.0